### 1. Imports and Configuration

In [ ]:
import math
import time

import numpy               as np
import torch
import torch.nn            as nn
import torch.nn.functional as F

torch.manual_seed(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# -----------------------------------------------------------------------
# Configuration.
# -----------------------------------------------------------------------

VOCAB      = 8192
MAX_BYTES  = 8
DIM        = 768
EMBED_DIM  = 256
SEQ_LEN    = 128
BATCH      = 32
N_LAYERS   = 3
N_STEPS    = 400
LR         = 3e-4

print(f'Device      : {device}')
print(f'Vocab       : {VOCAB}  |  Max bytes/token: {MAX_BYTES}')

### 2. BPE Token Byte Decomposition

In [ ]:
# -----------------------------------------------------------------------
# Simulate sp8192 token -> byte decomposition.
# Tokens 0-255 are single-byte identities. Higher token IDs tend to have
# more bytes (up to MAX_BYTES). Pseudo-random bytes seeded by token ID.
# -----------------------------------------------------------------------

token_to_bytes = {}
for t in range(VOCAB):
    if t < 256:
        token_to_bytes[t] = [t]
    else:
        n_bytes        = min(MAX_BYTES, 1 + (t - 256) // 1200 + 1)
        rng            = np.random.RandomState(t)
        token_to_bytes[t] = rng.randint(0, 256, size=n_bytes).tolist()

byte_targets_np = np.full((VOCAB, MAX_BYTES), 0, dtype=np.int64)
byte_lengths_np = np.zeros(VOCAB, dtype=np.int64)
for t in range(VOCAB):
    bts                 = token_to_bytes[t]
    byte_lengths_np[t]  = len(bts)
    for i, b in enumerate(bts):
        byte_targets_np[t, i] = b

byte_targets_t = torch.tensor(byte_targets_np, device=device)
byte_lengths_t = torch.tensor(byte_lengths_np, device=device)

avg_bytes = byte_lengths_t.float().mean().item()
print(f'Avg bytes per token : {avg_bytes:.2f}')
print(f'Byte distribution   : {[(byte_lengths_t == i).sum().item() for i in range(1, MAX_BYTES + 1)]}')

### 3. Synthetic Data

In [ ]:
# -----------------------------------------------------------------------
# Markov sequences with cluster structure. Tokens cluster into N_CLUSTERS
# groups with high within-cluster and lower cross-cluster transition probs.
# -----------------------------------------------------------------------

N_CLUSTERS         = 16
tokens_per_cluster = VOCAB // N_CLUSTERS
transition         = torch.zeros(VOCAB, VOCAB)

for c in range(N_CLUSTERS):
    s, e          = c * tokens_per_cluster, (c + 1) * tokens_per_cluster
    transition[s:e, s:e] = 1.0
    next_c        = (c + 1) % N_CLUSTERS
    ns, ne        = next_c * tokens_per_cluster, (next_c + 1) * tokens_per_cluster
    transition[s:e, ns:ne] = 0.3
    rand_c        = (c + 3) % N_CLUSTERS
    rs, re        = rand_c * tokens_per_cluster, (rand_c + 1) * tokens_per_cluster
    transition[s:e, rs:re] = 0.1

transition = transition / transition.sum(dim=-1, keepdim=True)


def generate_sequences(n_seqs, seq_len):
    seqs       = torch.zeros(n_seqs, seq_len + 1, dtype=torch.long)
    seqs[:, 0] = torch.randint(0, VOCAB, (n_seqs,))
    for t in range(seq_len):
        probs         = transition[seqs[:, t]]
        seqs[:, t + 1] = torch.multinomial(probs, 1).squeeze(-1)
    return seqs


train_data = generate_sequences(2000, SEQ_LEN).to(device)
val_data   = generate_sequences(500,  SEQ_LEN).to(device)
print(f'Train : {train_data.shape}  |  Val : {val_data.shape}')

### 4. Shared Transformer Components

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return F.rms_norm(x, (x.size(-1),)) * self.scale


class SimpleAttention(nn.Module):
    def __init__(self, dim, n_heads=8):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = dim // n_heads
        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.proj     = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, S, D  = x.shape
        qkv      = self.qkv(x).reshape(B, S, 3, self.n_heads, self.head_dim)
        q, k, v  = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]
        q, k, v  = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        scale    = 1.0 / math.sqrt(self.head_dim)
        attn     = (q @ k.transpose(-2, -1)) * scale
        mask     = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        attn     = F.softmax(attn.masked_fill(mask, float('-inf')), dim=-1)
        return self.proj((attn @ v).transpose(1, 2).reshape(B, S, D))


class SimpleMLP(nn.Module):
    def __init__(self, dim, mult=2):
        super().__init__()
        self.fc   = nn.Linear(dim, dim * mult, bias=False)
        self.proj = nn.Linear(dim * mult, dim, bias=False)
    def forward(self, x):
        return self.proj(F.relu(self.fc(x)).square())


class Block(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = SimpleAttention(dim)
        self.norm2 = RMSNorm(dim)
        self.mlp   = SimpleMLP(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

### 5. Model Definitions

In [ ]:
class StandardModel(nn.Module):
    '''BPE in, BPE out with embed_proj and tied output head.'''
    def __init__(self, vocab, dim, embed_dim, n_layers):
        super().__init__()
        self.tok_emb        = nn.Embedding(vocab, embed_dim)
        self.embed_proj     = nn.Linear(embed_dim, dim, bias=False)
        self.embed_proj_rev = nn.Linear(dim, embed_dim, bias=False)
        self.blocks         = nn.ModuleList([Block(dim) for _ in range(n_layers)])
        self.norm           = RMSNorm(dim)

    def forward(self, idx, targets):
        x      = self.embed_proj(self.tok_emb(idx))
        for block in self.blocks:
            x  = block(x)
        x      = self.norm(x)
        proj   = self.embed_proj_rev(x)
        logits = F.linear(proj, self.tok_emb.weight)
        loss   = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                 targets.reshape(-1), reduction='mean')
        return loss, logits

    def compute_bpb(self, idx, targets):
        '''Bits-per-byte weighted by token byte lengths.'''
        with torch.no_grad():
            loss_per_token = F.cross_entropy(
                self.forward(idx, targets)[1].reshape(-1, VOCAB),
                targets.reshape(-1), reduction='none'
            ).reshape(idx.shape)
            tgt_bytes  = byte_lengths_t[targets].float()
            bpb        = (loss_per_token.sum() / math.log(2.0)) / tgt_bytes.sum()
        return bpb.item()

    def param_summary(self):
        emb          = self.tok_emb.weight.numel()
        proj         = self.embed_proj.weight.numel() + self.embed_proj_rev.weight.numel()
        blocks       = sum(p.numel() for b in self.blocks for p in b.parameters())
        norm         = sum(p.numel() for p in self.norm.parameters())
        total        = emb + proj + blocks + norm
        fp_cost      = (emb + proj) * 2
        ternary_cost = blocks * 0.2
        return {
            'emb_params': emb, 'proj_params': proj, 'block_params': blocks,
            'total': total, 'fp_bytes': fp_cost, 'ternary_bytes': ternary_cost,
            'total_bytes': fp_cost + ternary_cost + norm * 2,
        }


class AsymmetricModel(nn.Module):
    '''
    BPE in (8k vocab), byte out (256 classes).
    K parallel heads each predict one byte position of the next token.
    Loss is the mean over all real byte positions.
    '''
    def __init__(self, vocab, dim, embed_dim, n_layers, max_bytes=MAX_BYTES):
        super().__init__()
        self.max_bytes  = max_bytes
        self.tok_emb    = nn.Embedding(vocab, embed_dim)
        self.embed_proj = nn.Linear(embed_dim, dim, bias=False)
        self.blocks     = nn.ModuleList([Block(dim) for _ in range(n_layers)])
        self.norm       = RMSNorm(dim)
        self.byte_heads = nn.ModuleList([nn.Linear(dim, 257, bias=False)
                                         for _ in range(max_bytes)])

    def _encode(self, idx):
        x = self.embed_proj(self.tok_emb(idx))
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

    def forward(self, idx, targets):
        x         = self._encode(idx)
        tgt_bytes = byte_targets_t[targets]
        tgt_lens  = byte_lengths_t[targets]
        total_loss  = torch.zeros((), device=x.device)
        total_bytes = 0
        for k in range(self.max_bytes):
            logits_k  = self.byte_heads[k](x)
            target_k  = tgt_bytes[:, :, k].clone()
            pad_mask  = (k >= tgt_lens)
            target_k[pad_mask] = 256
            loss_k    = F.cross_entropy(logits_k.reshape(-1, 257),
                                        target_k.reshape(-1), reduction='none')
            real_mask = ~pad_mask.reshape(-1)
            if real_mask.any():
                total_loss  = total_loss + loss_k[real_mask].sum()
                total_bytes += real_mask.sum().item()
        return total_loss / max(total_bytes, 1), None

    def compute_bpb(self, idx, targets):
        with torch.no_grad():
            loss, _ = self.forward(idx, targets)
        return loss.item() / math.log(2.0)

    def param_summary(self):
        emb          = self.tok_emb.weight.numel()
        proj         = self.embed_proj.weight.numel()
        blocks       = sum(p.numel() for b in self.blocks for p in b.parameters())
        heads        = sum(p.numel() for h in self.byte_heads for p in h.parameters())
        norm         = sum(p.numel() for p in self.norm.parameters())
        total        = emb + proj + blocks + heads + norm
        fp_cost      = (emb + proj + heads) * 2
        ternary_cost = blocks * 0.2
        return {
            'emb_params': emb, 'proj_params': proj, 'head_params': heads,
            'block_params': blocks, 'total': total,
            'fp_bytes': fp_cost, 'ternary_bytes': ternary_cost,
            'total_bytes': fp_cost + ternary_cost + norm * 2,
        }


class AsymmetricAutoregressive(nn.Module):
    '''
    BPE in, byte out with autoregressive byte decoder.
    A small GRU predicts bytes one at a time conditioned on the previous
    byte prediction. Training uses teacher forcing.
    '''
    def __init__(self, vocab, dim, embed_dim, n_layers, max_bytes=MAX_BYTES):
        super().__init__()
        self.max_bytes   = max_bytes
        self.tok_emb     = nn.Embedding(vocab, embed_dim)
        self.embed_proj  = nn.Linear(embed_dim, dim, bias=False)
        self.blocks      = nn.ModuleList([Block(dim) for _ in range(n_layers)])
        self.norm        = RMSNorm(dim)
        self.byte_embed  = nn.Embedding(258, 64)
        self.byte_rnn    = nn.GRU(dim + 64, 256, batch_first=True)
        self.byte_out    = nn.Linear(256, 257, bias=False)
        self.start_token = 257

    def forward(self, idx, targets):
        x = self.embed_proj(self.tok_emb(idx))
        for block in self.blocks:
            x = block(x)
        x         = self.norm(x)
        B, S, D   = x.shape
        tgt_bytes = byte_targets_t[targets]
        tgt_lens  = byte_lengths_t[targets]
        x_flat    = x.reshape(B * S, D)
        total_loss  = torch.zeros((), device=x.device)
        total_bytes = 0
        prev_byte   = torch.full((B * S,), self.start_token, dtype=torch.long, device=x.device)
        h           = None
        for k in range(self.max_bytes):
            byte_emb  = self.byte_embed(prev_byte)
            rnn_input = torch.cat([x_flat, byte_emb], dim=-1).unsqueeze(1)
            rnn_out, h = self.byte_rnn(rnn_input, h)
            logits    = self.byte_out(rnn_out.squeeze(1))
            target_k  = tgt_bytes[:, :, k].reshape(-1).clone()
            pad_mask  = (k >= tgt_lens).reshape(-1)
            target_k[pad_mask] = 256
            loss_k    = F.cross_entropy(logits, target_k, reduction='none')
            real_mask = ~pad_mask
            if real_mask.any():
                total_loss  = total_loss + loss_k[real_mask].sum()
                total_bytes += real_mask.sum().item()
            prev_byte          = target_k.clone()
            prev_byte[pad_mask] = 256
        return total_loss / max(total_bytes, 1), None

    def compute_bpb(self, idx, targets):
        with torch.no_grad():
            loss, _ = self.forward(idx, targets)
        return loss.item() / math.log(2.0)

    def param_summary(self):
        emb     = self.tok_emb.weight.numel()
        proj    = self.embed_proj.weight.numel()
        blocks  = sum(p.numel() for b in self.blocks for p in b.parameters())
        decoder = (self.byte_embed.weight.numel() +
                   sum(p.numel() for p in self.byte_rnn.parameters()) +
                   self.byte_out.weight.numel())
        norm    = sum(p.numel() for p in self.norm.parameters())
        total        = emb + proj + blocks + decoder + norm
        fp_cost      = (emb + proj + decoder) * 2
        ternary_cost = blocks * 0.2
        return {
            'emb_params': emb, 'proj_params': proj, 'decoder_params': decoder,
            'block_params': blocks, 'total': total,
            'fp_bytes': fp_cost, 'ternary_bytes': ternary_cost,
            'total_bytes': fp_cost + ternary_cost + norm * 2,
        }

### 6. Training and Evaluation

In [ ]:
def train_and_eval(model, name, n_steps=N_STEPS):
    model = model.to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    # -----------------------------------------------------------------------
    # Warmup passes to prime the optimizer state before timing.
    # -----------------------------------------------------------------------

    for _ in range(5):
        batch = train_data[torch.randint(0, len(train_data), (BATCH,), device=device)]
        loss, _ = model(batch[:, :-1], batch[:, 1:])
        loss.backward()
        opt.zero_grad()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    for _ in range(1, n_steps + 1):
        model.train()
        batch   = train_data[torch.randint(0, len(train_data), (BATCH,), device=device)]
        loss, _ = model(batch[:, :-1], batch[:, 1:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

    elapsed = time.perf_counter() - t0
    ms_step = elapsed / n_steps * 1000

    model.eval()
    val_bpb = model.compute_bpb(val_data[:, :-1], val_data[:, 1:])

    return {'name': name, 'val_bpb': val_bpb, 'ms_step': ms_step,
             'summary': model.param_summary()}

### 7. Run Experiments

In [ ]:
models = [
    ('Standard (tied, emb=256)', StandardModel(VOCAB, DIM, EMBED_DIM,  N_LAYERS)),
    ('Standard (tied, emb=128)', StandardModel(VOCAB, DIM, 128,        N_LAYERS)),
    ('Asym parallel (emb=256)',  AsymmetricModel(VOCAB, DIM, EMBED_DIM, N_LAYERS)),
    ('Asym parallel (emb=128)',  AsymmetricModel(VOCAB, DIM, 128,       N_LAYERS)),
    ('Asym parallel (emb=64)',   AsymmetricModel(VOCAB, DIM, 64,        N_LAYERS)),
    ('Asym autoreg (emb=256)',   AsymmetricAutoregressive(VOCAB, DIM, EMBED_DIM, N_LAYERS)),
    ('Asym autoreg (emb=128)',   AsymmetricAutoregressive(VOCAB, DIM, 128,       N_LAYERS)),
]

print(f'\n{"Model":<30} {"BPB":>8} {"ms/step":>8} {"Total":>10} {"FP bytes":>10} {"Tern bytes":>10} {"Est MB":>8}')
print('-' * 95)

for name, model in models:
    r      = train_and_eval(model, name)
    s      = r['summary']
    est_mb = s['total_bytes'] / 1e6
    print(f'{name:<30} {r["val_bpb"]:>8.4f} {r["ms_step"]:>7.1f}ms {s["total"]:>10,} {s["fp_bytes"]:>10,} {int(s["ternary_bytes"]):>10,} {est_mb:>7.2f}M')